In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q torchao==0.16.0
!pip install -q timm peft scikit-learn scipy

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.0 MB/s eta 0:00:00


In [ ]:
import os, json, pickle, random, time, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
import timm, torchvision, torchvision.models as tvm
from torchvision import datasets
from PIL import Image
from peft import LoraConfig, get_peft_model

PROJECT_DIR  = Path('/content/drive/MyDrive/Indian_Food_Research')
DATASET_PATH = PROJECT_DIR / 'datasets' / 'CompressedDataset'
SPLITS_DIR   = PROJECT_DIR / 'splits'
OUT_DIR      = PROJECT_DIR / 'results' / 'multiseed'
(OUT_DIR / 'weights').mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEEDS     = [0, 1, 2]
FRACTIONS = [1.0, 0.50, 0.25, 0.10]
METHODS   = ['linear_probe', 'lora', 'full_finetune', 'resnet50_scratch']
ABLATION  = ['lora_r4', 'lora_r16']
EPOCHS = {'resnet50_scratch': 15, 'linear_probe': 10, 'lora': 10, 'full_finetune': 10}
LR     = {'resnet50_scratch': 1e-3, 'linear_probe': 1e-3, 'lora': 1e-3, 'full_finetune': 1e-5}
WEIGHT_DECAY = 1e-4
for _m in ABLATION:
    EPOCHS[_m], LR[_m] = EPOCHS['lora'], LR['lora']
LORA_CFG = dict(r=8, alpha=16, dropout=0.1, target_modules=['qkv'], modules_to_save=['head'])
BATCH_CANDIDATES = [64, 48, 32]
SAVE_TRAINABLE_WEIGHTS = {'lora', 'linear_probe'}
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

full_ds = datasets.ImageFolder(str(DATASET_PATH))
class_names = full_ds.classes
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 100 and len(full_ds.samples) == 4578, 'Drive may be mid-sync. STOP.'
train_idx = pickle.load(open(SPLITS_DIR / 'train_idx.pkl', 'rb'))
val_idx   = pickle.load(open(SPLITS_DIR / 'val_idx.pkl',   'rb'))
test_idx  = pickle.load(open(SPLITS_DIR / 'test_idx.pkl',  'rb'))
assert not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx)) and not (set(train_idx) & set(val_idx))
print(len(train_idx), len(val_idx), len(test_idx), '| device', DEVICE)

versions = {'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__,
            'timm': timm.__version__, 'numpy': np.__version__}
import peft, sklearn, scipy
versions.update({'peft': peft.__version__, 'sklearn': sklearn.__version__, 'scipy': scipy.__version__})
try:
    import open_clip; versions['open_clip'] = open_clip.__version__
except Exception:
    versions['open_clip'] = 'not installed in this notebook (record it from notebook 02)'
(OUT_DIR / 'versions.json').write_text(json.dumps(versions, indent=2))
print(versions)

3127 678 678 | device cuda
{'python': '3.13.15', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'timm': '1.0.29', 'numpy': '2.1.3', 'peft': '0.21.0', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'open_clip': 'not installed in this notebook (record it from notebook 02)'}


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import hashlib

CACHE_DIR = PROJECT_DIR / 'cache'; CACHE_DIR.mkdir(exist_ok=True)
split_sig = hashlib.md5(json.dumps([list(map(int, train_idx)), list(map(int, val_idx)),
                                    list(map(int, test_idx))]).encode()).hexdigest()[:12]
CACHE_FILE = CACHE_DIR / f'images{IMG_SIZE}_u8_{split_sig}.pt'

def _load_one(idx):
    path, y = full_ds.samples[idx]
    with Image.open(path) as im:
        arr = np.asarray(im.convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
    return arr, y

def preload(indices, label='', workers=16):
    buf = torch.empty(len(indices), 3, IMG_SIZE, IMG_SIZE, dtype=torch.uint8)
    lab = torch.empty(len(indices), dtype=torch.long)
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for i, (arr, y) in enumerate(ex.map(_load_one, indices)):
            buf[i] = torch.from_numpy(arr.copy()).permute(2, 0, 1)
            lab[i] = y
            if (i + 1) % 1000 == 0: print(f'  {label} {i+1}/{len(indices)} ({time.time()-t0:.0f}s)')
    return buf, lab

if 'Xtr_u8' not in globals():
    if CACHE_FILE.exists():
        c = torch.load(CACHE_FILE)
        Xtr_u8, Ytr, Xva_u8, Yva, Xte_u8, Yte = (c[k] for k in ('Xtr', 'Ytr', 'Xva', 'Yva', 'Xte', 'Yte'))
        print('loaded cache from Drive:', CACHE_FILE.name)
    else:
        Xtr_u8, Ytr = preload(train_idx, 'train')
        Xva_u8, Yva = preload(val_idx, 'val')
        Xte_u8, Yte = preload(test_idx, 'test')
        torch.save(dict(Xtr=Xtr_u8, Ytr=Ytr, Xva=Xva_u8, Yva=Yva, Xte=Xte_u8, Yte=Yte), CACHE_FILE)
        print('saved cache to Drive:', CACHE_FILE.name)
all_t = np.array(full_ds.targets)
assert (Ytr.numpy() == all_t[list(train_idx)]).all() and (Yva.numpy() == all_t[list(val_idx)]).all() \
       and (Yte.numpy() == all_t[list(test_idx)]).all(), 'cached labels do not match the split'
print('cached:', Xtr_u8.shape, Xva_u8.shape, Xte_u8.shape)

MEAN_T = torch.tensor(MEAN, device=DEVICE).view(1, 3, 1, 1) * 255
STD_T  = torch.tensor(STD,  device=DEVICE).view(1, 3, 1, 1) * 255

def to_input(x_u8, train=False, gen=None):
    x = x_u8.to(DEVICE, non_blocking=True).float()
    if train:
        n = x.size(0)
        flip = torch.rand(n, device=DEVICE, generator=gen) < 0.5
        if flip.any():
            x[flip] = torch.flip(x[flip], dims=[3])
        r = lambda: 1 + (torch.rand(n, 1, 1, 1, device=DEVICE, generator=gen) * 0.4 - 0.2)
        x = x * r()
        g = x.mean(1, keepdim=True)
        x = g + (x - g) * r()
        m = x.mean(dim=(1, 2, 3), keepdim=True)
        x = m + (x - m) * r()
        x = x.clamp_(0, 255)
    return ((x - MEAN_T) / STD_T).contiguous(memory_format=torch.channels_last)

loaded cache from Drive: images224_u8_343f07cdbe89.pt
cached: torch.Size([3127, 3, 224, 224]) torch.Size([678, 3, 224, 224]) torch.Size([678, 3, 224, 224])


In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def build_model(method):
    if method == 'resnet50_scratch':
        m = tvm.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        return m
    m = timm.create_model('vit_base_patch14_dinov2', pretrained=True, num_classes=NUM_CLASSES, img_size=IMG_SIZE)
    if method == 'full_finetune':
        return m
    if method == 'linear_probe':
        for n, p in m.named_parameters():
            p.requires_grad = n.startswith('head')
        return m
    if method.startswith('lora'):
        r = int(method.split('_r')[1]) if '_r' in method else LORA_CFG['r']
        cfg = LoraConfig(r=r, lora_alpha=LORA_CFG['alpha'], target_modules=LORA_CFG['target_modules'],
                         lora_dropout=LORA_CFG['dropout'], modules_to_save=LORA_CFG['modules_to_save'])
        return get_peft_model(m, cfg)
    raise ValueError(method)

def label_budget_positions(fraction, seed):
    if fraction >= 1.0:
        return torch.arange(len(train_idx))
    rng = random.Random(seed)
    by_class = {}
    for pos, y in enumerate(Ytr.tolist()):
        by_class.setdefault(y, []).append(pos)
    sel = []
    for y, lst in sorted(by_class.items()):
        k = min(len(lst), max(1, round(fraction * len(lst))))
        lst = lst[:]; rng.shuffle(lst); sel.extend(lst[:k])
    rng.shuffle(sel)
    return torch.tensor(sel)

@torch.no_grad()
def predict_cached(model, X_u8, bs=128):
    model.eval(); out = []
    for i in range(0, len(X_u8), bs):
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            out.append(model(to_input(X_u8[i:i+bs])).argmax(1).cpu())
    return torch.cat(out).numpy()

def run_one(method, fraction, seed):
    set_seed(seed)
    pos = label_budget_positions(fraction, seed)
    Xs, Ys = Xtr_u8[pos], Ytr[pos]
    model = build_model(method).to(DEVICE).to(memory_format=torch.channels_last)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    epochs = EPOCHS[method]
    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR[method], weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    cpu_gen = torch.Generator(); cpu_gen.manual_seed(seed)
    best_val, best_ep, best_state, batch = -1.0, -1, None, None
    y_val = Yva.numpy()
    t_start = time.time()
    for ep in range(epochs):
        for cand in ([batch] if batch else BATCH_CANDIDATES):
            try:
                model.train()
                perm = torch.randperm(len(Xs), generator=cpu_gen)
                for i in range(0, len(Xs), cand):
                    sel = perm[i:i + cand]
                    if len(sel) < 2: continue
                    xb = to_input(Xs[sel], train=True, gen=gen)
                    yb = Ys[sel].to(DEVICE, non_blocking=True)
                    opt.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
                        loss = crit(model(xb), yb)
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                batch = cand
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); print(f'OOM at batch {cand}, retrying smaller')
                if cand == BATCH_CANDIDATES[-1]: raise
        sched.step()
        pv = predict_cached(model, Xva_u8)
        va = float((pv == y_val).mean())
        if va > best_val:
            best_val, best_ep = va, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f'    epoch {ep+1}/{epochs} val={va:.4f} ({time.time()-t_start:.0f}s)')
    model.load_state_dict(best_state)
    pv = predict_cached(model, Xva_u8)
    pt = predict_cached(model, Xte_u8)
    meta = dict(method=method, fraction=fraction, seed=seed, n_train=int(len(Xs)), epochs=epochs, lr=LR[method],
                weight_decay=WEIGHT_DECAY, batch_size=batch, best_epoch=best_ep, best_val_acc=best_val,
                test_acc=float((pt == Yte.numpy()).mean()), trainable_params=int(trainable), total_params=int(total),
                lora=({**LORA_CFG, 'r': int(method.split('_r')[1])} if '_r' in method else LORA_CFG) if method.startswith('lora') else None, minutes=round((time.time() - t_start) / 60, 2))
    wpath = OUT_DIR / 'weights' / f'{method}__f{int(round(fraction*100)):03d}__s{seed}.pt'
    if method in ('full_finetune', 'resnet50_scratch'):
        if fraction >= 1.0:
            torch.save({k: v.cpu() for k, v in model.state_dict().items()}, wpath)
    elif method in SAVE_TRAINABLE_WEIGHTS or method.startswith('lora'):
        keep = {n for n, p in model.named_parameters() if p.requires_grad}
        torch.save({k: v.cpu() for k, v in model.state_dict().items() if k in keep}, wpath)
    del model, opt; torch.cuda.empty_cache()
    return pt, pv, meta

In [ ]:
for seed in SEEDS:
    for fraction in FRACTIONS:
        for method in METHODS + (ABLATION if fraction >= 1.0 else []):
            tag = f'{method}__f{int(round(fraction*100)):03d}__s{seed}'
            out = OUT_DIR / f'{tag}.npz'
            if out.exists():
                print('SKIP', tag); continue
            print(f'\n=== {tag} ===')
            pt, pv, meta = run_one(method, fraction, seed)
            np.savez(out, test_pred=pt, test_true=Yte.numpy(), val_pred=pv, val_true=Yva.numpy(),
                     meta=json.dumps(meta))
            with open(OUT_DIR / 'runs.jsonl', 'a') as f: f.write(json.dumps(meta) + '\n')
            print(f'-> test {meta["test_acc"]*100:.2f}% | best val {meta["best_val_acc"]*100:.2f}% (epoch {meta["best_epoch"]}) | '
                  f'trainable {meta["trainable_params"]:,}/{meta["total_params"]:,} | {meta["minutes"]} min')
print('done')

SKIP linear_probe__f100__s0
SKIP lora__f100__s0
SKIP full_finetune__f100__s0
SKIP resnet50_scratch__f100__s0
SKIP lora_r4__f100__s0
SKIP lora_r16__f100__s0
SKIP linear_probe__f050__s0
SKIP lora__f050__s0
SKIP full_finetune__f050__s0
SKIP resnet50_scratch__f050__s0
SKIP linear_probe__f025__s0
SKIP lora__f025__s0
SKIP full_finetune__f025__s0
SKIP resnet50_scratch__f025__s0
SKIP linear_probe__f010__s0
SKIP lora__f010__s0
SKIP full_finetune__f010__s0
SKIP resnet50_scratch__f010__s0
SKIP linear_probe__f100__s1
SKIP lora__f100__s1
SKIP full_finetune__f100__s1

=== resnet50_scratch__f100__s1 ===
    epoch 1/15 val=0.0147 (14s)
    epoch 2/15 val=0.0649 (25s)
    epoch 3/15 val=0.1268 (36s)
    epoch 4/15 val=0.1298 (47s)
    epoch 5/15 val=0.1991 (58s)
    epoch 6/15 val=0.2227 (69s)
    epoch 7/15 val=0.2906 (81s)
    epoch 8/15 val=0.3702 (92s)
    epoch 9/15 val=0.4307 (103s)
    epoch 10/15 val=0.4661 (114s)
    epoch 11/15 val=0.4985 (125s)
    epoch 12/15 val=0.5221 (136s)
    epoch 13/

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

    epoch 1/10 val=0.9410 (32s)
    epoch 2/10 val=0.9528 (64s)
    epoch 3/10 val=0.9631 (96s)
    epoch 4/10 val=0.9690 (127s)
    epoch 5/10 val=0.9690 (159s)
    epoch 6/10 val=0.9705 (190s)
    epoch 7/10 val=0.9705 (222s)
    epoch 8/10 val=0.9705 (253s)
    epoch 9/10 val=0.9705 (284s)
    epoch 10/10 val=0.9705 (316s)
-> test 95.58% | best val 97.05% (epoch 6) | trainable 224,356/86,026,184 | 5.35 min

=== lora_r16__f100__s1 ===
    epoch 1/10 val=0.9440 (32s)
    epoch 2/10 val=0.9558 (63s)
    epoch 3/10 val=0.9617 (95s)
    epoch 4/10 val=0.9690 (126s)
    epoch 5/10 val=0.9690 (157s)
    epoch 6/10 val=0.9735 (189s)
    epoch 7/10 val=0.9690 (220s)
    epoch 8/10 val=0.9690 (252s)
    epoch 9/10 val=0.9690 (283s)
    epoch 10/10 val=0.9690 (314s)
-> test 95.87% | best val 97.35% (epoch 6) | trainable 666,724/86,468,552 | 5.33 min

=== linear_probe__f050__s1 ===
    epoch 1/10 val=0.8319 (8s)
    epoch 2/10 val=0.9071 (17s)
    epoch 3/10 val=0.9395 (25s)
    epoch 4/10 val=

In [ ]:
rows = [json.loads(l) for l in open(OUT_DIR / 'runs.jsonl')] if (OUT_DIR / 'runs.jsonl').exists() else []
if rows:
    d = pd.DataFrame(rows)
    print(d.pivot_table(index=['method', 'fraction'], columns='seed', values='test_acc').mul(100).round(2).to_string())

seed                           0      1      2
method           fraction                     
full_finetune    0.10      57.23  51.62  55.01
                 0.25      84.66  82.45  82.30
                 0.50      90.86  90.71  92.33
                 1.00      92.77  95.28  95.43
linear_probe     0.10      78.32  77.88  80.53
                 0.25      88.50  88.79  87.91
                 0.50      92.48  92.92  92.63
                 1.00      94.10  94.10  94.25
lora             0.10      78.47  78.91  82.45
                 0.25      89.97  91.59  90.41
                 0.50      93.66  94.25  94.54
                 1.00      95.43  96.02  96.31
lora_r16         1.00      95.72  95.87  95.43
lora_r4          1.00      95.28  95.58  95.87
resnet50_scratch 0.10       6.34  12.83  10.91
                 0.25      23.60  21.39  17.70
                 0.50      43.07  41.59  45.87
                 1.00      60.32  64.90  66.08
